In [10]:
import csv

with open("../data/games.csv", encoding="utf-8-sig", newline="") as f:
    r = csv.reader(f)
    header = next(r)
    row = next(r)

print("Số tên cột:", len(header), "| Số giá trị ở dòng đầu:", len(row))
for h, v in zip(header[:12], row[:12]):
    print(f"{h!r:30} -> {v!r}")

Số tên cột: 39 | Số giá trị ở dòng đầu: 40
'AppID'                        -> '2539430'
'Name'                         -> 'Black Dragon Mage Playtest'
'Release date'                 -> 'Aug 1, 2023'
'Estimated owners'             -> '0 - 0'
'Peak CCU'                     -> '0'
'Required age'                 -> '0'
'Price'                        -> '0.0'
'DiscountDLC count'            -> '0'
'About the game'               -> '0'
'Supported languages'          -> ''
'Full audio languages'         -> '[]'
'Reviews'                      -> '[]'


In [ ]:
import csv
import pandas as pd
from pathlib import Path

# ==================================================
# 1. Đường dẫn
# Notebook nằm trong thư mục notebooks/, dataset nằm trong data/
# ==================================================
DATA_PATH = Path("../data/games.csv")
OUTPUT_PATH = Path("../data/games_reduced.csv")
ENCODING = "utf-8-sig"  # utf-8-sig đọc được cả file có BOM lẫn không có

# ==================================================
# 2. Đọc header gốc và sửa lỗi header
# File raw của bộ Steam Games thường có header "DiscountDLC count"
# (dính 2 cột "Discount" và "DLC count" làm một). Header có 39 tên
# nhưng mỗi dòng dữ liệu có 40 giá trị, nên pandas hiểu nhầm cột đầu
# là index và đẩy toàn bộ dữ liệu lệch sang trái 1 cột.
# ==================================================
with open(DATA_PATH, "r", encoding=ENCODING, newline="") as f:
    reader = csv.reader(f)
    raw_header = next(reader)
    first_row = next(reader)

fixed_header = []
for col in raw_header:
    if col == "DiscountDLC count":
        fixed_header.extend(["Discount", "DLC count"])
    else:
        fixed_header.append(col)

print("Số cột trong header gốc :", len(raw_header))
print("Số cột trong header sửa :", len(fixed_header))
print("Số giá trị ở dòng dữ liệu đầu tiên:", len(first_row))

if len(fixed_header) != len(first_row):
    raise ValueError(
        "Số tên cột vẫn không khớp số giá trị trong dòng dữ liệu. "
        "Hãy mở file bằng trình soạn thảo văn bản và kiểm tra lại header."
    )

# ==================================================
# 3. Đọc dữ liệu với tên cột đã sửa
# names=fixed_header + header=0: bỏ dòng header cũ, dùng tên cột mới
# ==================================================
df = pd.read_csv(
    DATA_PATH,
    names=fixed_header,
    header=0,
    encoding=ENCODING,
    low_memory=False,
)

print("\nKích thước ban đầu:", df.shape)
print(
    "Dung lượng ban đầu:",
    round(df.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB",
)

# ==================================================
# 4. Kiểm tra dữ liệu có bị lệch cột không
# AppID phải là số, Release date phải trông giống ngày tháng.
# ==================================================
appid_numeric_rate = pd.to_numeric(df["AppID"], errors="coerce").notna().mean()
print(f"\nTỷ lệ AppID là số: {appid_numeric_rate:.2%}")
display_cols = [c for c in ["AppID", "Name", "Release date", "Price"] if c in df.columns]
print(df[display_cols].head())

if appid_numeric_rate < 0.99:
    raise ValueError(
        "Cột AppID không phải số, dữ liệu vẫn đang bị lệch cột. "
        "Hãy kiểm tra lại header và dữ liệu của file raw."
    )

# ==================================================
# 5. Chọn các cột cần thiết cho nghiên cứu
# ==================================================
keep_columns = [
    "AppID",
    "Name",
    "Release date",
    "Estimated owners",
    "Peak CCU",
    "Required age",
    "Price",
    "Discount",
    "DLC count",
    "Positive",
    "Negative",
    "User score",
    "Metacritic score",
    "Recommendations",
    "Average playtime forever",
    "Average playtime two weeks",
    "Median playtime forever",
    "Median playtime two weeks",
    "Developers",
    "Publishers",
    "Categories",
    "Genres",
    "Tags",
    "Windows",
    "Mac",
    "Linux",
]

available_columns = [col for col in keep_columns if col in df.columns]
missing_columns = [col for col in keep_columns if col not in df.columns]

df_reduced = df.loc[:, available_columns].copy()

# Đổi AppID thành id để các bước sau dùng df["id"]
if "AppID" in df_reduced.columns and "id" not in df_reduced.columns:
    df_reduced = df_reduced.rename(columns={"AppID": "id"})

# ==================================================
# 6. Báo cáo các cột không được giữ
# ==================================================
removed_columns = [col for col in df.columns if col not in available_columns]

print("\nSố cột ban đầu:", len(df.columns))
print("Số cột được giữ:", len(available_columns))
print("Số cột không giữ:", len(removed_columns))
print("\nCác cột không có trong file:", missing_columns)
print("\nCác cột bị loại khỏi dataset phân tích:")
print(removed_columns)

# ==================================================
# 7. So sánh dung lượng bộ nhớ
# ==================================================
original_memory = df.memory_usage(deep=True).sum()
reduced_memory = df_reduced.memory_usage(deep=True).sum()

print("\nDung lượng DataFrame ban đầu:", round(original_memory / 1024**2, 2), "MB")
print("Dung lượng DataFrame rút gọn:", round(reduced_memory / 1024**2, 2), "MB")

if original_memory > 0:
    reduction_percent = (original_memory - reduced_memory) / original_memory * 100
    print("Mức giảm bộ nhớ:", round(reduction_percent, 2), "%")

# ==================================================
# 8. Lưu file mới, không ghi đè file gốc
# ==================================================
df_reduced.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print("\nĐã lưu dataset rút gọn tại:", OUTPUT_PATH)
print("Kích thước dataset rút gọn:", df_reduced.shape)

Số cột trong header gốc : 39
Số cột trong header sửa : 40
Số giá trị ở dòng dữ liệu đầu tiên: 40

Kích thước ban đầu: (8, 40)
Dung lượng ban đầu: 0.04 MB

Tỷ lệ AppID là số: 100.00%
     AppID                                   Name  Release date  Price
0  2539430             Black Dragon Mage Playtest   Aug 1, 2023   0.00
1   496350  Supipara - Chapter 1 Spring Has Come!  Jul 29, 2016   5.24
2  1034400      Mystery Solitaire The Black Raven   May 6, 2019   4.99
3  3292190            버튜버 파라노이아 - Vtuber Paranoia  Oct 31, 2024   8.99
4  3631080                          Maze Quest VR  Apr 24, 2025   4.99

Số cột ban đầu: 40
Số cột được giữ: 26
Số cột không giữ: 14

Các cột không có trong file: []

Các cột bị loại khỏi dataset phân tích:
['About the game', 'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url', 'Support email', 'Metacritic url', 'Score rank', 'Achievements', 'Notes', 'Screenshots', 'Movies']

Dung lượng DataFrame ban đầu: 0.04 MB
